# Phase 3-4: Data Loading & Cleaning

Recovers a malformed Matrix Software POS report export into a clean, analysis-ready transaction table. The raw export repeats report boilerplate on every line and buries the real transaction fields inside it (currency values use European-style formatting, quantities use accounting-style negative parentheses, and ~3.7% of lines are truncated junk). This notebook parses the real records out, converts all the numeric formatting, and runs a full cleaning/validation pass.

## Parse the malformed report export

In [2]:

import re
import csv

pattern = re.compile(
    r'"Avg /Kg","([^"]*)","([^"]*)","(Invoices|Credit Notes)","(\d{4}/\d{2}/\d{2})",(\d+),"([^"]*)","([^"]*)",(\(?-?[\d,]*\.?\d+\)?),(\(?-?[\d,]*\.?\d+\)?),"([^"]*)","([^"]*)"'
)

def parse_num(s):
    s = s.strip()
    neg = s.startswith('(') or s.startswith('-')
    s = s.replace('(', '').replace(')', '').replace('-', '').replace(',', '')
    val = float(s) if s else 0.0
    return -val if neg else val

def parse_money(s):
    s = s.strip()
    if not s:
        return None
    neg = s.startswith('-')
    s = s.replace('R', '').replace('-', '').strip()
    if not s:
        return 0.0
    parts = s.split(',')
    if len(parts) == 1:
        val = float(parts[0]) if parts[0] else 0.0
    else:
        cents = parts[-1]
        integer = ''.join(parts[:-1])
        val = float(f"{integer}.{cents}")
    return -val if neg else val

in_path = "../data/polokwane_sales.csv"
out_path = "../data/parsed_raw.csv"
unmatched_path = "../outputs/unmatched_lines.txt"

fields = ["date","doc_type","doc_number","debtor_code","debtor_name","product_code","product_desc","qty","mass_kg","value_zar","avg_price_per_kg"]

n_total = 0
n_matched = 0
n_unmatched = 0

with open(in_path, "r", encoding="latin-1", errors="replace") as fin, \
     open(out_path, "w", newline="", encoding="utf-8") as fout, \
     open(unmatched_path, "w", encoding="utf-8") as funmatch:
    writer = csv.writer(fout)
    writer.writerow(fields)
    for line in fin:
        n_total += 1
        m = pattern.search(line)
        if not m:
            n_unmatched += 1
            if n_unmatched <= 20:
                funmatch.write(line)
            continue
        n_matched += 1
        product_code, product_desc, doc_type, date, doc_num, debtor_code, debtor_name, qty, mass, value_str, avgkg_str = m.groups()
        row = [
            date,
            doc_type,
            doc_num,
            debtor_code,
            debtor_name,
            product_code,
            product_desc,
            parse_num(qty),
            parse_num(mass),
            parse_money(value_str),
            parse_money(avgkg_str),
        ]
        writer.writerow(row)

print("total lines:", n_total)
print("matched:", n_matched)
print("unmatched:", n_unmatched)

total lines: 582395
matched: 560912
unmatched: 21483


## Load parsed data & run the full cleaning pass

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/parsed_raw.csv', dtype={'debtor_code': str, 'product_code': str})

report = []
report.append(f"Raw parsed rows: {len(df):,}")

# --- 1. Parse date ---
df['date'] = pd.to_datetime(df['date'], format='%Y/%m/%d', errors='coerce')
n_bad_date = df['date'].isna().sum()
report.append(f"Rows with unparseable date: {n_bad_date}")
df = df.dropna(subset=['date'])

# --- 2. Exact duplicate rows ---
n_dupes = df.duplicated().sum()
report.append(f"Exact duplicate rows: {n_dupes}")
df = df.drop_duplicates()

# --- 3. Whitespace / text cleanup ---
for col in ['debtor_code', 'debtor_name', 'product_code', 'product_desc', 'doc_type']:
    df[col] = df[col].astype(str).str.strip()

# --- 4. Missing values ---
report.append("\nMissing values per column:")
report.append(df.isna().sum().to_string())

# --- 5. Data type sanity ---
report.append(f"\ndoc_number range: {df['doc_number'].min()} - {df['doc_number'].max()}")
report.append(f"date range: {df['date'].min().date()} - {df['date'].max().date()}")

# --- 6. Consistency: Credit Notes should generally have negative qty/value; flag inconsistencies ---
credit = df[df['doc_type'] == 'Credit Notes']
inv = df[df['doc_type'] == 'Invoices']
report.append(f"\nInvoices: {len(inv):,} rows | Credit Notes: {len(credit):,} rows")
odd_credit = credit[(credit['qty'] >= 0) & (credit['value_zar'] >= 0)]
report.append(f"Credit Notes with non-negative qty & value (unexpected): {len(odd_credit)}")
odd_invoice = inv[(inv['qty'] < 0) | (inv['value_zar'] < 0)]
report.append(f"Invoices with negative qty or value (returns mislabelled as invoices?): {len(odd_invoice)}")

# --- 7. Zero / negative price sanity ---
report.append(f"\nRows with value_zar == 0: {(df['value_zar']==0).sum()}")
report.append(f"Rows with qty == 0: {(df['qty']==0).sum()}")

# --- 8. Outlier check on value_zar ---
q99 = df['value_zar'].quantile(0.99)
q001 = df['value_zar'].quantile(0.001)
report.append(f"\nvalue_zar 0.1th pct: {q001:.2f}, 99th pct: {q99:.2f}, max: {df['value_zar'].max():.2f}, min: {df['value_zar'].min():.2f}")

# --- 9. Distinct products / debtors ---
report.append(f"\nDistinct products: {df['product_code'].nunique()}")
report.append(f"Distinct debtors: {df['debtor_code'].nunique()}")
report.append(f"Distinct doc_numbers: {df['doc_number'].nunique()} (rows: {len(df)}) -- doc_number is not unique per row since one doc can have multiple product lines")

# --- 10. Derive unit price check (value/qty vs avg_price_per_kg validity) ---
report.append(f"\nRows where mass_kg==0 but avg_price_per_kg != 0: {((df['mass_kg']==0) & (df['avg_price_per_kg']!=0)).sum()}")

with open('../outputs/cleaning_report.txt', 'w') as f:
    f.write("\n".join(report))

print("\n".join(report))
print("\nFinal cleaned shape:", df.shape)

df.to_csv('../data/polokwane_sales_clean.csv', index=False)

Raw parsed rows: 560,912
Rows with unparseable date: 0
Exact duplicate rows: 0

Missing values per column:
date                0
doc_type            0
doc_number          0
debtor_code         0
debtor_name         0
product_code        0
product_desc        0
qty                 0
mass_kg             0
value_zar           0
avg_price_per_kg    0

doc_number range: 300331 - 30000052
date range: 2023-07-01 - 2026-08-25

Invoices: 560,840 rows | Credit Notes: 72 rows
Credit Notes with non-negative qty & value (unexpected): 0
Invoices with negative qty or value (returns mislabelled as invoices?): 372

Rows with value_zar == 0: 24554
Rows with qty == 0: 8431

value_zar 0.1th pct: 0.00, 99th pct: 1739.80, max: 48068.64, min: -14258.27

Distinct products: 1617
Distinct debtors: 101
Distinct doc_numbers: 8565 (rows: 560912) -- doc_number is not unique per row since one doc can have multiple product lines

Rows where mass_kg==0 but avg_price_per_kg != 0: 0

Final cleaned shape: (560912, 11)
